In [1]:
# === Imports ===
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
import os

# === OpenAI Client Setup ===
load_dotenv()
client = OpenAI()


c:\Users\verye\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def format_sidebar(state):
    """
    Builds the sidebar's Markdown text from a token-tracking state dict
    with keys: total_tokens, summary, last_input_tokens,
    last_output_tokens, last_summarization_tokens.
    """
    lines = ["### 📊 Conversation Summary"]
    if state["summary"]:
        lines.append(state["summary"])
    else:
        lines.append("*Summary will appear here after a few messages.*")

    lines.append("### 💰 Token Usage")
    if state["last_input_tokens"] > 0:
        lines.append("**Last Chat Interaction:**")
        lines.append(f"- Chat Input (sent to API): {state['last_input_tokens']} tokens")
        lines.append(f"- Chat Output (from API): {state['last_output_tokens']} tokens")
        lines.append(f"- Chat Total: {state['last_input_tokens'] + state['last_output_tokens']} tokens")

        if state["last_summarization_tokens"] > 0:
            lines.append(f"- **Summarization API Call:** {state['last_summarization_tokens']} tokens")
            lines.append("*(Summarization uses gpt-4o-mini, which is much cheaper)*")

        lines.append("---")

    lines.append(f"**Cumulative Total:** {state['total_tokens']} tokens")
    lines.append("*Lower tokens = lower costs!*")

    return "\n\n".join(lines)
# Formats sidebar for summarization


In [3]:
def summarize_conversation(messages):
    """
    Summarize the entire conversation using OpenAI's API.
    Returns the summary text and total tokens used for this API call.

    Uses gpt-4o-mini (a smaller, faster model) instead of gpt-4o because:
    - Summarization is a simpler task that doesn't need the most powerful model
    - gpt-4o-mini is much cheaper (~1/15th the cost) and faster
    - It's still excellent at summarization tasks

    This is a common pattern: use powerful models for user-facing responses,
    and smaller models for background tasks like summarization.
    """
    conversation_text = "\n".join(f"{msg['role']}: {msg['content']}" for msg in messages)
    summary_text = f"Summarize the following conversation concisely:\n{conversation_text}\nSummary:"

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": summary_text}],
        max_completion_tokens=750
    )

    summarization_tokens = response.usage.prompt_tokens + response.usage.completion_tokens
    return response.choices[0].message.content, summarization_tokens
# Summarizes messages with a cap on tokens and returns the summary + token count


In [4]:
def check_and_summarize(history, token_state):
    """
    Every 3 user messages, replace the older conversation history with
    a short summary, to keep the context sent to the API small.

    Parameters:
    history (list): The conversation so far, ending in the assistant's
                     latest reply.
    token_state (dict): Tracks total_tokens, summary, last_input_tokens,
                         last_output_tokens, and last_summarization_tokens.

    Returns:
    tuple: (history, token_state, sidebar_markdown)
    """
    token_state = dict(token_state)
    token_state["last_summarization_tokens"] = 0  # Reset for this turn

    user_message_count = sum(1 for msg in history if msg["role"] == "user")

    # Summarize every 3 user messages (after the assistant has responded)
    if user_message_count > 0 and user_message_count % 3 == 0:
    
        # Pre-calculated variables (no complex indexing needed!)
        messages_to_summarize = history[:-1]        # Everything except the latest
        current_assistant_message = history[-1]      # Just the latest message

        summary_text, summarization_tokens = summarize_conversation(messages_to_summarize)
       

        # Store the summary and token usage
        token_state["summary"] = summary_text
        token_state["last_summarization_tokens"] = summarization_tokens
        token_state["total_tokens"] += summarization_tokens

        # Replace long history with summary + current message
        history = [
            {"role": "system", "content": f"🔄 This is a summary of the conversation so far: {summary_text}"},
            current_assistant_message,
        ]

    return history, token_state, format_sidebar(token_state)


In [20]:
system_message = {"role": "system", "content": "You are being used for customer service training. You are an angry customer who comes into the mobile phone store with a device that won't turn on and the user (store employee) must diagnose the problem. Stay in character as the customer at all times, You should stay on topic, you are never the employee in this situation. Pick a realistic root problem (example: dead battery, water damage, a damaged charging port, a frozen software update) and only reveal clues when the employee asks good diagnostic questions. Calm down gradually if the employee is empathetic, professional and helpful, get more frustrated if they are dismissive, keep the replies short as if you were a real customer"}
initial_greeting = {"role": "assistant", "content": "Hey, I have been having some trouble with my phone."}

In [23]:
def user_submit(message, history):
    # Add the user's message to the history, and clear the textbox
    history = history + [{"role": "user", "content": message}]
    return history, ""

def bot_respond(history, token_state):
    token_state = dict(token_state)
    history = [system_message] + history   
    stream = client.chat.completions.create(
        model="gpt-4o",
        messages=history,
        stream=True,
        stream_options={"include_usage": True}  # Get token usage from the streaming response
    )

    history = history + [{"role": "assistant", "content": ""}]

    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            history[-1]["content"] += chunk.choices[0].delta.content
            yield history, token_state

        if hasattr(chunk, "usage") and chunk.usage:
            token_state["last_input_tokens"] = chunk.usage.prompt_tokens
            token_state["last_output_tokens"] = chunk.usage.completion_tokens
            token_state["total_tokens"] += chunk.usage.prompt_tokens + chunk.usage.completion_tokens
            yield history, token_state

def reset_conversation():
    fresh_history = [system_message, initial_greeting]
    fresh_state = {
        "total_tokens": 0,
        "summary": "",
        "last_input_tokens": 0,
        "last_output_tokens": 0,
        "last_summarization_tokens": 0,
    }
    return fresh_history, fresh_state, format_sidebar(fresh_state)

gr.close_all()

initial_history = [{"role": "assistant", "content": "Hey, I have been having some problems with my phone"}]
initial_state = {
    "total_tokens": 0,
    "summary": "",
    "last_input_tokens": 0,
    "last_output_tokens": 0,
    "last_summarization_tokens": 0,
}

with gr.Blocks(title="My AI Chatbot with Smart Context Management", fill_width=True) as demo:
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🤖")
        with gr.Column(scale=4):
            gr.Markdown("### My AI Chatbot with Smart Context Management")
            gr.Markdown("*Powered by OnePiece*")

    gr.Markdown("---")

    token_state = gr.State(initial_state)

    with gr.Sidebar():
        sidebar_md = gr.Markdown(format_sidebar(initial_state))
        reset_btn = gr.Button("🔄 Reset Conversation")

    chatbot = gr.Chatbot(value=initial_history, label="Chat", height=400)
    msg = gr.Textbox(placeholder="Type your message here...", label="Your message")

    msg.submit(user_submit, inputs=[msg, chatbot], outputs=[chatbot, msg]).then(
        bot_respond, inputs=[chatbot, token_state], outputs=[chatbot, token_state]
    ).then(
        check_and_summarize, inputs=[chatbot, token_state], outputs=[chatbot, token_state, sidebar_md]
    )

    reset_btn.click(reset_conversation, outputs=[chatbot, token_state, sidebar_md])

demo.launch()


Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
